# Visual analysis of titles (2023)

In [1]:
import random

import pandas as pd
import numpy as np
from tqdm import tqdm
import json

## 1. Read embedding file

In [2]:
df = pd.read_csv('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors3.title', sep='\t')
df.head(5)

,ent_id:token,ent_emb:float_seq
0,884509,0.010017607 0.052419856 -0.033521876 0.0235467...
1,561856,-0.032739636 -0.1026613 0.032374747 0.07612567...
2,239749,0.062343705 0.13869876 0.22980133 -0.050816707...
3,55030,-0.064253606 0.03547375 0.09745385 0.15618598 ...
4,1277121,0.12226398 0.11615175 0.060607553 0.045587018 ...


## 2. Preprocess embeddings

In [3]:
df.rename(columns={"ent_emb:float_seq": "emb", "ent_id:token": "id"}, inplace=True)
df["emb"] = df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))
df.head(5)

,id,emb
0,884509,"[0.010017607, 0.052419856, -0.033521876, 0.023..."
1,561856,"[-0.032739636, -0.1026613, 0.032374747, 0.0761..."
2,239749,"[0.062343705, 0.13869876, 0.22980133, -0.05081..."
3,55030,"[-0.064253606, 0.03547375, 0.09745385, 0.15618..."
4,1277121,"[0.12226398, 0.11615175, 0.060607553, 0.045587..."


## 3. Fit KNN with static embeddings

In [4]:
from sklearn.neighbors import NearestNeighbors

X = np.vstack(df["emb"].values)
knn = NearestNeighbors(n_neighbors=10, metric='cosine')
knn.fit(X)

NearestNeighbors(metric='cosine', n_neighbors=10)

## 4. Take random item and find its neighbours

In [5]:
import random

n_ids = df["id"].shape[0]

while True:
    random_idx = random.choice(range(n_ids))

    if sum(X[random_idx]) != 0:
        break

orig_id = df["id"].iloc[random_idx]

dist, nearest_ids = knn.kneighbors([X[random_idx]])

print(orig_id)

338387


## 5. Convert into correct metadata

In [6]:
ids = list(nearest_ids[0])
distances = dist[0]

print(ids)

[581786, 1219237, 1166880, 1016588, 222065, 643796, 296360, 1067623, 757493, 979526]


In [7]:
import csv

data = {}
orig_title = ''
with open('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.item') as f:
    reader = csv.reader(f, delimiter='\t')
    _ = next(reader)
    for line in reader:
        idx = int(line[0])
        title = line[1]

        if idx == orig_id:
            orig_title = title
            continue

        if idx not in ids:
            continue

        data[idx] = title

## 6. Visual analysis

### 6.1 Original title

In [8]:
print(orig_title)

HOPITU Camping Chair, Compact Outdoor Chair, Light Durable Aluminum Alloy Folding Chair, 360 Rotating Swivel Chair with Carry Bag, Aircraft Grade Aluminum, Supports 302lbs


### 6.2 Supposedly similar titles (sorted by distance)

In [9]:
for idx in ids:
    print(f"{data[idx]}\n")

Octobermoon Original Second Gerneration 180°view Panoramic full face Snorkel Mask,with anti-fog anti-leak snorkeling Design,See More water world Larger Viewing Area (Full Black II, Large/Extra Large)

Bike Phone Holder Bag Detachable Waterproof Bicycle Phone Mount Handlebar Bags with 360°Rotation TPU Touch Screen Shockproof Breathable Interior Cycling Pouch Compatible with Phones under 6.5"

Hydration Pack Backpack HuanLang 2L Lightweight Water Backpack Marathon Running Water Vest with 1.5L Water Bladder for Outdoors Sports Walking Hiking Cycling Climbing Hunting for Men Women Kids (Blue)

Light & Motion Seca 1700 Enduro - 6 Cell Pack

Lifetime Freestyle Hard Shell Paddleboard with Paddle, 9'8"/X-Large, Glacier Blue

AR500-Targets Square

UCO Mini Candle Lantern - Anodized Green - Lightest Tealight A-LTN-STD

Rest Your Mind Calm Your Heart Vinyl Wall Decals Quotes Sayings Words Art Decor Lettering Vinyl Wall Art Inspirational Uplifting

Vew-Do Zone Balance Board

Silicone Swim Cap,Swim